> GitHub export: personal-photo image outputs omitted; code, aggregate metrics and diagnostic tables retained. Local cohort has prior project exposure and is not a fresh independent holdout.

# Local road damage evaluation — Runs 17 and 14

The following code audits all 190 reviewed local images, verifies D00/D10/D20/D40 labels, and evaluates the adopted fine-tuned YOLOv8s (Run 17) against the earlier COCO-initialized YOLO11s (Run 14). Both use the same images and 640-pixel inference. Confidence 0.46 was frozen before local testing.

Results include detection mAP, per-class box precision/recall/F1, image-level Macro-F1, confusion matrices, confidence curves and a comparison chart. Each output section includes a short interpretation. JSON, CSV, plots and a ZIP archive are saved in Kaggle outputs. No training or threshold optimization is performed on these images.

Labels were created with AI assistance and reviewed by the team. Annotation errors and similar views of the same road may affect the results. This compares complete pipelines, not architecture alone.

In [2]:
from IPython.display import display, Markdown
from pathlib import Path
import json, hashlib, shutil, sys, subprocess
import numpy as np
import pandas as pd
from PIL import Image
import yaml

display(Markdown("# RoadScan — Independent local test\nEvaluate the adopted fine-tuned Run 17 and the earlier COCO-initialized Run 14 on the same 190 local images. No training or threshold tuning is performed. Labels were generated with an assistant model and reviewed by the team; annotation quality remains a limitation. This compares two pipelines, not architecture alone."))
display(Markdown("## 1. Audit images and labels\nVerify image decoding, four-class mapping, box geometry and duplicate image hashes. Preserve healthy images with empty labels. Stage an unchanged test copy with portable paths."))
CLASSES=['D00','D10','D20','D40']
ART=Path('/kaggle/working/local_test_results'); ART.mkdir(exist_ok=True)
yamls=list(Path('/kaggle/input').glob('**/roadscan-local-test-v1/**/data.yaml'))
assert len(yamls)==1, yamls
src=yamls[0].parent
meta=yaml.safe_load(yamls[0].read_text()); assert meta['names']==CLASSES, meta
images=sorted((src/'test/images').glob('*'))
images=[p for p in images if p.suffix.lower() in ['.jpg','.jpeg','.png','.webp']]
assert len(images)==190, len(images)
ROOT=Path('/kaggle/working/local_test_data'); (ROOT/'test/images').mkdir(parents=True,exist_ok=True); (ROOT/'test/labels').mkdir(parents=True,exist_ok=True)
rows=[]; truth=[]; gt=[]
for p in images:
    label=src/'test/labels'/f'{p.stem}.txt'
    assert label.exists(), f'Missing label: {p.name}'
    with Image.open(p) as im:
        im.load(); w,h=im.size; pixelhash=hashlib.sha256(im.convert('RGB').tobytes()).hexdigest()
    boxes=[]; present=np.zeros(4,dtype=int)
    for line in label.read_text().splitlines():
        if not line.strip(): continue
        v=list(map(float,line.split())); assert len(v)==5, (label,v)
        c,x,y,bw,bh=v; assert c==int(c) and 0<=c<4 and 0<bw<=1 and 0<bh<=1
        assert min(x-bw/2,y-bh/2)>=-0.001 and max(x+bw/2,y+bh/2)<=1.001, (label,v)
        boxes.append([c,(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h]); present[int(c)]=1
    gt.append(np.array(boxes).reshape(-1,5)); truth.append(present)
    rows.append(dict(file=p.name,width=w,height=h,boxes=len(boxes),pixel_sha256=pixelhash,label_sha256=hashlib.sha256(label.read_bytes()).hexdigest()))
    shutil.copy2(p,ROOT/'test/images'/p.name); shutil.copy2(label,ROOT/'test/labels'/label.name)
manifest=pd.DataFrame(rows); manifest.to_csv(ART/'manifest.csv',index=False)
assert not manifest.pixel_sha256.duplicated().any(), 'Exact duplicate local images: review before evaluation.'
DATA=ROOT/'data.yaml'; DATA.write_text(yaml.safe_dump(dict(path=str(ROOT),train='test/images',val='test/images',test='test/images',names=CLASSES)))
truth=np.array(truth)
display(pd.DataFrame({'class':CLASSES,'images_with_damage':truth.sum(0)}))
display(Markdown(f"**Audit result:** {len(images)} readable images, {sum(manifest.boxes)} boxes and {(manifest.boxes==0).sum()} healthy/background images. All images remain test-only. YAML train/val aliases only satisfy the loader; no training is called. Exact duplicates are checked; near-duplicate views may still reduce effective sample size."))

display(Markdown("## 2. Freeze evaluation settings and checkpoints\nUse 640 pixels, NMS IoU 0.7 and at most 300 detections. Report detection mAP using confidence 0.001 to retain the confidence ranking, and operational box precision/recall/F1 at the previously frozen 0.46 threshold with same-class IoU 0.5 matching. These are different metrics. Run 14 uses the same operational threshold for a common operating-point comparison, not its own optimized threshold."))
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU required'
b14=next(Path('/kaggle/input').glob('**/14-roadscan-team-v2-training/artifacts_team_v2/best.pt'))
b17=next(Path('/kaggle/input').glob('**/17-roadscan-gentle-finetuning/run17_model_comparison/run17_best_finetuned.pt'))
weights={'run17_finetuned':b17,'run14_yolo11s':b14}
protocol=dict(imgsz=640,nms_iou=0.7,max_det=300,operational_conf=0.46,matching_iou=0.5,map_conf=0.001,local_images=190,training=False,threshold_tuning=False,annotation='AI-assisted, team-reviewed',ultralytics=ultralytics.__version__,weights={k:dict(path=str(v),sha256=hashlib.sha256(v.read_bytes()).hexdigest()) for k,v in weights.items()})
(ART/'protocol.json').write_text(json.dumps(protocol,indent=2))
display(Markdown("**Checkpoint check:** Run 17 uses run17_best_finetuned.pt, never selected_candidate.pt. Run 14 uses its trained best.pt; it was initialized from COCO, not trained entirely from scratch. Both file hashes are saved."))

def iou_one(a,bs):
    lo=np.maximum(a[:2],bs[:,:2]); hi=np.minimum(a[2:],bs[:,2:]); inter=np.maximum(hi-lo,0).prod(1)
    return inter/np.maximum((a[2:]-a[:2]).prod()+(bs[:,2:]-bs[:,:2]).prod(1)-inter,1e-9)
allrows=[]; summaries=[]
for name,path in weights.items():
    display(Markdown(f"## 3. Evaluate {name}\nMeasure the unchanged checkpoint against the reviewed local labels. Save mAP plots, fixed-threshold detection counts, image-level metrics and prediction examples."))
    model=YOLO(str(path)); assert [model.names[i] for i in range(4)]==CLASSES, model.names
    det=model.val(data=str(DATA),split='test',imgsz=640,batch=8,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=True,project=str(ART),name=name+'_map',exist_ok=True)
    scores=np.zeros((190,4)); counts=np.zeros((4,3),dtype=int); errors=[]
    lookup={p.name:i for i,p in enumerate(images)}
    for res in model.predict(source=str(ROOT/'test/images'),imgsz=640,conf=0.46,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
        i=lookup[Path(res.path).name]; predictions=res.boxes.data.cpu().numpy(); g=gt[i]
        for c in range(4):
            ps=predictions[predictions[:,5]==c]; ps=ps[np.argsort(-ps[:,4])]; gs=g[g[:,0]==c,1:]; used=set(); tp=0
            if len(ps): scores[i,c]=ps[:,4].max()
            for p in ps:
                if len(gs):
                    overlaps=iou_one(p[:4],gs)
                    for j in used: overlaps[j]=-1
                    j=int(overlaps.argmax())
                    if overlaps[j]>=0.5: used.add(j); tp+=1
            counts[c]+=np.array([tp,len(ps)-tp,len(gs)-tp])
        if i<12: res.save(filename=str(ART/f'{name}_example_{i:02d}.jpg'))
        errors.append(dict(file=Path(res.path).name,predicted_boxes=len(predictions),true_boxes=len(g)))
    from sklearn.metrics import precision_recall_fscore_support
    ip,ir,iff,support=precision_recall_fscore_support(truth,scores>=0.46,average=None,zero_division=0)
    cr=[]
    for c in range(4):
        tp,fp,fn=map(int,counts[c]); p=tp/max(tp+fp,1); r=tp/max(tp+fn,1); f=2*p*r/max(p+r,1e-9)
        cr.append(dict(model=name,cls=CLASSES[c],TP=tp,FP=fp,FN=fn,box_precision=p,box_recall=r,box_f1=f,image_precision=float(ip[c]),image_recall=float(ir[c]),image_f1=float(iff[c]),support=int(truth[:,c].sum())))
    frame=pd.DataFrame(cr); frame.to_csv(ART/f'{name}_per_class.csv',index=False); display(frame)
    pd.DataFrame(errors).to_csv(ART/f'{name}_image_counts.csv',index=False)
    np.savez(ART/f'{name}_image_predictions.npz',truth=truth,scores=scores)
    total=counts.sum(0); tp,fp,fn=map(int,total); p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    summary=dict(model=name,mAP50=float(det.box.map50),mAP50_95=float(det.box.map),box_precision_046=p,box_recall_046=r,box_f1_046=2*p*r/max(p+r,1e-9),image_macro_f1_046=float(iff.mean()),TP=tp,FP=fp,FN=fn)
    summaries.append(summary); allrows+=cr
    (ART/f'{name}_results.json').write_text(json.dumps(summary,indent=2))
    display(Markdown(f"**Results:** mAP50 {summary['mAP50']:.1%}; mAP50–95 {summary['mAP50_95']:.1%}. At frozen confidence 0.46, box precision {p:.1%}, recall {r:.1%} and F1 {summary['box_f1_046']:.1%}. Image-level Macro-F1 {iff.mean():.1%} measures class presence, not localization. False positives: {fp}; missed boxes: {fn}. No threshold is chosen from these test results."))
comparison=pd.DataFrame(summaries); comparison.to_csv(ART/'comparison.csv',index=False); display(comparison)
import matplotlib.pyplot as plt
comparison.set_index('model')[['mAP50','mAP50_95','box_f1_046','image_macro_f1_046']].plot.bar(rot=0,ylim=(0,1),figsize=(10,5)); plt.tight_layout(); plt.savefig(ART/'comparison.png'); plt.show()
(ART/'results.json').write_text(json.dumps(dict(protocol=protocol,models=summaries,per_class=allrows),indent=2))
report='## Local test interpretation\n\n'+comparison.to_markdown(index=False)+'\n\nThis is a paired local test of two fixed pipelines. Architecture and initialization differ, so differences cannot be attributed to architecture alone. Run 17 remains the preselected adopted model. Local labels were AI-assisted and human-reviewed; errors in the labels and correlated views limit conclusions. No training, test-driven model selection or confidence tuning was performed. Small per-class supports should be interpreted cautiously.'
(ART/'RESULTS_INTERPRETATION.md').write_text(report); display(Markdown(report))
shutil.make_archive('/kaggle/working/local_test_results','zip',ART)
print('LOCAL_TEST_COMPLETE',json.dumps(summaries))


# RoadScan — Independent local test
Evaluate the adopted fine-tuned Run 17 and the earlier COCO-initialized Run 14 on the same 190 local images. No training or threshold tuning is performed. Labels were generated with an assistant model and reviewed by the team; annotation quality remains a limitation. This compares two pipelines, not architecture alone.

## 1. Audit images and labels
Verify image decoding, four-class mapping, box geometry and duplicate image hashes. Preserve healthy images with empty labels. Stage an unchanged test copy with portable paths.

,class,images_with_damage
0,D00,89
1,D10,53
2,D20,33
3,D40,43


**Audit result:** 190 readable images, 333 boxes and 56 healthy/background images. All images remain test-only. YAML train/val aliases only satisfy the loader; no training is called. Exact duplicates are checked; near-duplicate views may still reduce effective sample size.

## 2. Freeze evaluation settings and checkpoints
Use 640 pixels, NMS IoU 0.7 and at most 300 detections. Report detection mAP using confidence 0.001 to retain the confidence ranking, and operational box precision/recall/F1 at the previously frozen 0.46 threshold with same-class IoU 0.5 matching. These are different metrics. Run 14 uses the same operational threshold for a common operating-point comparison, not its own optimized threshold.

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


... (8 outputs left out)

**Results:** mAP50 32.1%; mAP50–95 13.2%. At frozen confidence 0.46, box precision 61.2%, recall 12.3% and F1 20.5%. Image-level Macro-F1 41.7% measures class presence, not localization. False positives: 26; missed boxes: 292. No threshold is chosen from these test results.

,model,mAP50,mAP50_95,box_precision_046,box_recall_046,box_f1_046,image_macro_f1_046,TP,FP,FN
0,run17_finetuned,0.262451,0.131487,0.50000,0.174174,0.258352,0.502049,58,58,275
1,run14_yolo11s,0.320965,0.132172,0.61194,0.123123,0.205000,0.416895,41,26,292


## Local test interpretation

| model           |    mAP50 |   mAP50_95 |   box_precision_046 |   box_recall_046 |   box_f1_046 |   image_macro_f1_046 |   TP |   FP |   FN |
|:----------------|---------:|-----------:|--------------------:|-----------------:|-------------:|---------------------:|-----:|-----:|-----:|
| run17_finetuned | 0.262451 |   0.131487 |             0.5     |         0.174174 |     0.258352 |             0.502049 |   58 |   58 |  275 |
| run14_yolo11s   | 0.320965 |   0.132172 |             0.61194 |         0.123123 |     0.205    |             0.416895 |   41 |   26 |  292 |

This is a paired local test of two fixed pipelines. Architecture and initialization differ, so differences cannot be attributed to architecture alone. Run 17 remains the preselected adopted model. Local labels were AI-assisted and human-reviewed; errors in the labels and correlated views limit conclusions. No training, test-driven model selection or confidence tuning was performed. Small per-class supports should be interpreted cautiously.

LOCAL_TEST_COMPLETE [{"model": "run17_finetuned", "mAP50": 0.2624511953497888, "mAP50_95": 0.13148693083466007, "box_precision_046": 0.5, "box_recall_046": 0.17417417417417416, "box_f1_046": 0.2583518930957684, "image_macro_f1_046": 0.5020489755122438, "TP": 58, "FP": 58, "FN": 275}, {"model": "run14_yolo11s", "mAP50": 0.32096510562564323, "mAP50_95": 0.13217164929068886, "box_precision_046": 0.6119402985074627, "box_recall_046": 0.12312312312312312, "box_f1_046": 0.205, "image_macro_f1_046": 0.4168954996186117, "TP": 41, "FP": 26, "FN": 292}]


In [3]:
# Diagnostic review: explanation before computation; interpretation follows outputs.
display(Markdown("## 4. Diagnose missed damage without changing the model\nCompare frozen-confidence predictions with labels, inspect low-confidence candidates at 0.05 for diagnosis only, and create side-by-side review sheets. Original labels, weights and test scores remain unchanged. This is post-test exploratory analysis; any resulting model or threshold change needs a new independent test set."))
from PIL import ImageDraw, ImageOps
D=ART/'diagnostic_review'; D.mkdir(exist_ok=True)
review_model=YOLO(str(b17)); pred_cache={}; diagnostic=[]
for rr in review_model.predict(source=str(ROOT/'test/images'),imgsz=640,conf=0.05,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
    idx=lookup[Path(rr.path).name]; ps=rr.boxes.data.cpu().numpy(); pred_cache[idx]=ps
    g=gt[idx]; matched=set()
    for pp in sorted(ps[ps[:,4]>=0.46],key=lambda z:-z[4]):
        choices=[j for j in range(len(g)) if j not in matched and int(g[j,0])==int(pp[5])]
        if choices:
            ov=iou_one(pp[:4],g[choices,1:]); k=int(ov.argmax())
            if ov[k]>=0.5: matched.add(choices[k])
    w,h=Image.open(images[idx]).size
    for j,gg in enumerate(g):
        same=ps[ps[:,5]==gg[0]]; high=same[same[:,4]>=0.46]; other=ps[(ps[:,5]!=gg[0])&(ps[:,4]>=0.46)]
        ov=iou_one(gg[1:],same[:,:4]) if len(same) else np.array([])
        oh=iou_one(gg[1:],high[:,:4]) if len(high) else np.array([])
        oo=iou_one(gg[1:],other[:,:4]) if len(other) else np.array([])
        if j in matched: reason='matched'
        elif len(oh) and oh.max()>=0.5: reason='one_to_one_competition'
        elif len(ov) and ov.max()>=0.5: reason='low_confidence_candidate'
        elif len(oo) and oo.max()>=0.5: reason='other_class_overlap'
        elif len(ov) and ov.max()>=0.1: reason='localization_or_label_extent'
        else: reason='no_same_class_overlap'
        bw,bh=gg[3]-gg[1],gg[4]-gg[2]
        diagnostic.append(dict(index=idx,file=images[idx].name,label_index=j,cls=CLASSES[int(gg[0])],reason=reason,area_fraction=bw*bh/(w*h),thin_side_at_640=min(bw,bh)*640/max(w,h),best_same_iou=float(ov.max()) if len(ov) else 0))
dfdiag=pd.DataFrame(diagnostic); dfdiag.to_csv(D/'box_diagnosis.csv',index=False)
display(pd.crosstab(dfdiag.cls,dfdiag.reason)); display(dfdiag.groupby('reason')[['area_fraction','thin_side_at_640']].median())
# Stratified visual review: first two files from each class/reason plus healthy examples.
chosen=[]
for _,ggg in dfdiag[dfdiag.reason!='matched'].groupby(['cls','reason'],sort=True):
    for idx in ggg['index'].drop_duplicates().head(2):
        if int(idx) not in chosen: chosen.append(int(idx))
chosen=chosen[:20]+[i for i,g in enumerate(gt) if len(g)==0][:4]
chosen=list(dict.fromkeys(chosen))
def panel(idx,pred=False):
    im=Image.open(images[idx]).convert('RGB'); im.thumbnail((440,340)); canvas=Image.new('RGB',(460,390),'white'); canvas.paste(im,(10,35)); dr=ImageDraw.Draw(canvas)
    w,h=Image.open(images[idx]).size; sx=im.width/w; sy=im.height/h
    dr.text((10,5),('PRED conf >= 0.46' if pred else 'REVIEWED LABELS')+' | '+str(idx),fill='black')
    boxes=pred_cache[idx] if pred else gt[idx]
    for bb in boxes:
        if pred and bb[4]<0.46: continue
        c=int(bb[5] if pred else bb[0]); box=bb[:4] if pred else bb[1:]
        xy=[10+box[0]*sx,35+box[1]*sy,10+box[2]*sx,35+box[3]*sy]; dr.rectangle(xy,outline=('red' if pred else 'blue'),width=2)
        dr.text((xy[0],max(20,xy[1]-11)),CLASSES[c]+(f' {bb[4]:.2f}' if pred else ''),fill=('red' if pred else 'blue'))
    dr.text((10,375),images[idx].name[:65],fill='black'); return canvas
for page,start in enumerate(range(0,len(chosen),4)):
    sheet=Image.new('RGB',(920,390*len(chosen[start:start+4])),'white')
    for row,idx in enumerate(chosen[start:start+4]):
        sheet.paste(panel(idx),(0,row*390)); sheet.paste(panel(idx,True),(460,row*390))
    sheet.save(D/f'review_sheet_{page+1}.jpg'); display(sheet)
queue=pd.DataFrame([dict(index=i,file=images[i].name,review_status='pending',action='',notes='',scene_group='') for i in chosen]); queue.to_csv(D/'annotation_review_queue.csv',index=False)
counts=dfdiag.reason.value_counts().to_dict()
display(Markdown('**Diagnostic interpretation:** '+str(counts)+'\n\nThese are overlap-based hypotheses, not confirmed causes. Check blue label boxes and red model boxes in the review sheets. Correct missing/wrong class labels and box extent only from visible road evidence, not to match predictions. Do not tune confidence on this set. No new test split is created from already examined images. Collect new scenes, including healthy roads; keep all views of one scene together.'))
(D/'diagnosis.json').write_text(json.dumps(dict(counts=counts,reviewed_visually=False,queue_images=chosen,post_test_exploratory=True,original_test_unchanged=True),indent=2))
shutil.make_archive('/kaggle/working/local_test_diagnostic_review','zip',D)
print('DIAGNOSTIC_COMPLETE',counts)


## 4. Diagnose missed damage without changing the model
Compare frozen-confidence predictions with labels, inspect low-confidence candidates at 0.05 for diagnosis only, and create side-by-side review sheets. Original labels, weights and test scores remain unchanged. This is post-test exploratory analysis; any resulting model or threshold change needs a new independent test set.

reason,localization_or_label_extent,low_confidence_candidate,matched,no_same_class_overlap,other_class_overlap
cls,,,,,
D00,18,13,19,96,9
D10,9,11,4,56,11
D20,5,11,17,5,0
D40,13,6,18,12,0


,area_fraction,thin_side_at_640
reason,,
localization_or_label_extent,0.149693,149.579832
low_confidence_candidate,0.054893,80.317460
matched,0.074714,110.158730
no_same_class_overlap,0.046733,64.126984
other_class_overlap,0.077117,82.492997


**Diagnostic interpretation:** {'no_same_class_overlap': 169, 'matched': 58, 'localization_or_label_extent': 45, 'low_confidence_candidate': 41, 'other_class_overlap': 20}

These are overlap-based hypotheses, not confirmed causes. Check blue label boxes and red model boxes in the review sheets. Correct missing/wrong class labels and box extent only from visible road evidence, not to match predictions. Do not tune confidence on this set. No new test split is created from already examined images. Collect new scenes, including healthy roads; keep all views of one scene together.

DIAGNOSTIC_COMPLETE {'no_same_class_overlap': 169, 'matched': 58, 'localization_or_label_extent': 45, 'low_confidence_candidate': 41, 'other_class_overlap': 20}


In [4]:
display(Markdown("## 5. Visual review and next-step decision\nRecord a preliminary visual audit and a development protocol. This does not edit the dataset or retrain the model."))
notes={19:'Large D00 label spans a broad road region; inspect original crack extent. Prediction differs in extent/class. Do not approve solely because a crack exists.',68:'Long narrow labeled crack and broader predicted D00 box overlap poorly; box convention needs review. Small additional labels need full-resolution verification.',48:'Visible diagonal crack is labeled D00 but not detected at frozen confidence: genuine missed-damage candidate.',53:'Many overlapping D00/D10 labels and only one detected region. Review instance boundaries and road-relative direction; model also misses visible crack regions.',88:'Multiple small D00/D10 labels; no high-confidence detections. Review visibility and class direction against road travel, not screen orientation.',50:'D10 label is near a strong lighting/shadow boundary. Inspect full resolution to distinguish an actual crack from the illumination edge.',130:'Overlapping D40 boxes cover adjacent broken asphalt regions. Inspect whether these are separate potholes or duplicate labels, and separate alligator cracking from cavities.',145:'Broad D20 label surrounds visible network cracking with no high-confidence prediction: missed-damage candidate.'}
visual=pd.DataFrame([dict(index=i,file=images[i].name,review_status='preliminary_visual_review',notes=n,requires_full_resolution_confirmation=True) for i,n in notes.items()]); visual.to_csv(D/'visual_review_notes.csv',index=False)
plan='''# Diagnostic findings and recovery plan

Run 17 at confidence 0.46 matches 58/333 annotated boxes. Of 275 unmatched labels: 169 have no same-class overlap >=0.1 even among candidates retained at confidence 0.05; 45 have partial same-class overlap (0.1-0.5); 41 have same-class IoU >=0.5 only below operational confidence; 20 overlap a different predicted class. These categories depend on labels and are hypotheses, not proof of annotation error or a new performance estimate.

D00 and D10 account for 223/275 misses (81.1%). Lower confidence alone cannot address most misses. Visual sampling shows genuine missed crack candidates alongside label-extent, overlapping-box and possible shadow-boundary cases. No causal explanation is established from these samples alone.

1. Audit local annotation rules against RDD2022 conventions. Define D00/D10 relative to road direction. Use tight axis-aligned boxes around full damage instances, consistently separate individual cracks/network cracking/cavities, and remove duplicate labels only after independent visual confirmation. Preserve healthy images. A second reviewer should check the flagged cases at full resolution without trying to imitate model predictions.
2. Keep the original local-test-v1 and initial test results as a historical snapshot. If the 190 images influence training, confidence, augmentation, resolution or checkpoint selection, explicitly relabel them as development data in the next dataset version. Do not claim a randomly held-out subset of these already inspected images is a fresh test set.
3. Group local photos by the actual road damage/location; all views of the same scene stay in one partition. Scene grouping requires team location knowledge and is not inferable reliably from filenames alone. If fine-tuning locally, split corrected development scenes into train/validation before training and document the split seed and groups.
4. Collect a separate sealed local test from NEW scenes never used for development, including healthy roads and diverse lighting, cameras, distances and all four classes. Freeze the model, confidence and evaluation settings before opening its results. Report per-class support, metrics, missed damage and false positives. Do not promise high accuracy from a small sample.
5. Only after the label audit, choose one targeted experiment: local fine-tuning if domain mismatch remains, or higher-resolution validation if thin distant cracks dominate. Select using development validation, never the new test. Retain run17 as the historical adopted checkpoint until a documented replacement is validated.

No training or label modification has been performed in this diagnostic notebook. Eight scene examples received preliminary visual review; this is not a full ground-truth audit. The new independent test cannot be created from the already examined 190 images.
'''
(D/'RECOVERY_PLAN.md').write_text(plan); display(Markdown(plan)); display(visual[['index','notes']])
state=json.loads((D/'diagnosis.json').read_text()); state.update(preliminary_visual_review_indices=list(notes),full_annotation_audit_complete=False); (D/'diagnosis.json').write_text(json.dumps(state,indent=2))
shutil.make_archive('/kaggle/working/local_test_diagnostic_review','zip',D)
shutil.make_archive('/kaggle/working/local_test_results','zip',ART)
print('REVIEW_AND_PLAN_SAVED')


## 5. Visual review and next-step decision
Record a preliminary visual audit and a development protocol. This does not edit the dataset or retrain the model.

# Diagnostic findings and recovery plan

Run 17 at confidence 0.46 matches 58/333 annotated boxes. Of 275 unmatched labels: 169 have no same-class overlap >=0.1 even among candidates retained at confidence 0.05; 45 have partial same-class overlap (0.1-0.5); 41 have same-class IoU >=0.5 only below operational confidence; 20 overlap a different predicted class. These categories depend on labels and are hypotheses, not proof of annotation error or a new performance estimate.

D00 and D10 account for 223/275 misses (81.1%). Lower confidence alone cannot address most misses. Visual sampling shows genuine missed crack candidates alongside label-extent, overlapping-box and possible shadow-boundary cases. No causal explanation is established from these samples alone.

1. Audit local annotation rules against RDD2022 conventions. Define D00/D10 relative to road direction. Use tight axis-aligned boxes around full damage instances, consistently separate individual cracks/network cracking/cavities, and remove duplicate labels only after independent visual confirmation. Preserve healthy images. A second reviewer should check the flagged cases at full resolution without trying to imitate model predictions.
2. Keep the original local-test-v1 and initial test results as a historical snapshot. If the 190 images influence training, confidence, augmentation, resolution or checkpoint selection, explicitly relabel them as development data in the next dataset version. Do not claim a randomly held-out subset of these already inspected images is a fresh test set.
3. Group local photos by the actual road damage/location; all views of the same scene stay in one partition. Scene grouping requires team location knowledge and is not inferable reliably from filenames alone. If fine-tuning locally, split corrected development scenes into train/validation before training and document the split seed and groups.
4. Collect a separate sealed local test from NEW scenes never used for development, including healthy roads and diverse lighting, cameras, distances and all four classes. Freeze the model, confidence and evaluation settings before opening its results. Report per-class support, metrics, missed damage and false positives. Do not promise high accuracy from a small sample.
5. Only after the label audit, choose one targeted experiment: local fine-tuning if domain mismatch remains, or higher-resolution validation if thin distant cracks dominate. Select using development validation, never the new test. Retain run17 as the historical adopted checkpoint until a documented replacement is validated.

No training or label modification has been performed in this diagnostic notebook. Eight scene examples received preliminary visual review; this is not a full ground-truth audit. The new independent test cannot be created from the already examined 190 images.


,index,notes
0,19,Large D00 label spans a broad road region; ins...
1,68,Long narrow labeled crack and broader predicte...
2,48,Visible diagonal crack is labeled D00 but not ...
3,53,Many overlapping D00/D10 labels and only one d...
4,88,Multiple small D00/D10 labels; no high-confide...
5,50,D10 label is near a strong lighting/shadow bou...
6,130,Overlapping D40 boxes cover adjacent broken as...
7,145,Broad D20 label surrounds visible network crac...


REVIEW_AND_PLAN_SAVED
